# AMEX Enterprise Credit Risk Platform
## Notebook 16 — Production Architecture: As-Built Stack, Capacity Sizing & Target Topology
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Architecture**. Notebook 16 of 18. Depends on Notebook 01 only; Notebooks 09 (real latency/throughput), 11 (real container build), 12 (real monitoring thresholds) and 14 (financial figures) are cross-referenced opportunistically if present.

**Two kinds of content, clearly separated throughout.** This notebook fuses two very different things and never blurs them: (1) a real, live-scanned inventory of what this platform has **actually built** on disk this run -- the Dockerfile, the FastAPI service, the CI/CD YAML, the monitoring job -- verified by checking for the real files, not claimed; and (2) a **target production topology and capacity plan**, which necessarily requires business-scale inputs this dataset cannot supply (expected request volume, an SLA latency target, infrastructure unit costs) -- every one of those is a stated, editable **ASSUMPTION**, and every capacity number derived from them is computed live from Notebook 09's real, measured throughput and latency where that notebook has run, never estimated from thin air.

**Deliverables:** `technology_stack_inventory.csv` (real, as-built), `capacity_sizing.json` (real measurement + stated assumptions), `production_architecture_diagram.png`, `production_readiness_checklist.csv`, and `Production_Architecture_Report.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOK 01 (EVERYTHING ELSE OPTIONAL)
# =============================================================================
import os
import sys
import json
import math
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebook 01 (Everything Else Optional)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first -- "
                             f"this notebook reads its pillar directory map.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)

PROD_ARCH_DIR = PILLAR_DIRS["production_architecture"]
PROD_ARCH_DIR.mkdir(parents=True, exist_ok=True)

NB09_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_09_summary.json"
NB11_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_11_summary.json"
NB12_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_12_summary.json"
NB14_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_14_summary.json"

NB09_SUMMARY = json.load(open(NB09_SUMMARY_PATH, encoding="utf-8")) if NB09_SUMMARY_PATH.exists() else None
NB11_SUMMARY = json.load(open(NB11_SUMMARY_PATH, encoding="utf-8")) if NB11_SUMMARY_PATH.exists() else None
NB12_SUMMARY = json.load(open(NB12_SUMMARY_PATH, encoding="utf-8")) if NB12_SUMMARY_PATH.exists() else None
NB14_SUMMARY = json.load(open(NB14_SUMMARY_PATH, encoding="utf-8")) if NB14_SUMMARY_PATH.exists() else None

# MLOps model_registry.json carries the real, measured latency/throughput from
# Notebook 09's own benchmark -- read it directly for the champion's latest entry.
MLOPS_DIR = PILLAR_DIRS["mlops"]
REGISTRY_PATH = MLOPS_DIR / "model_registry.json"
LATEST_REGISTRY_ENTRY = None
if REGISTRY_PATH.exists():
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        _registry = json.load(f)
    if _registry.get("entries"):
        LATEST_REGISTRY_ENTRY = max(_registry["entries"], key=lambda e: (e.get("model_name", ""), e.get("version", 0)))

print(f"Notebook 09 (MLOps) summary  : {'found' if NB09_SUMMARY else 'not found -- skipping (not required)'}")
print(f"Notebook 11 (Docker) summary : {'found' if NB11_SUMMARY else 'not found -- skipping (not required)'}")
print(f"Notebook 12 (Monitoring) summary: {'found' if NB12_SUMMARY else 'not found -- skipping (not required)'}")
print(f"Notebook 14 (Executive Reports) summary: {'found' if NB14_SUMMARY else 'not found -- skipping (not required)'}")
print(f"Real model registry latency/throughput: {'found' if LATEST_REGISTRY_ENTRY else 'not found -- Notebook 09 has not been run yet'}")
print(f"Production architecture artifacts will be written under: {PROD_ARCH_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

print("(Reporting only -- this notebook is I/O-bound documentation/planning work, not thread-parallelized.)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)
print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: TECHNOLOGY STACK -- AS-BUILT, LIVE FILE-EXISTENCE EVIDENCE
# =============================================================================
_section("SECTION 3: Technology Stack -- As-Built, Live File-Existence Evidence")

# --- Every row below is checked against a real file this platform's own
#     notebooks would have written -- not a claimed capability list. ---
_evidence_checks = [
    {"layer": "Data Engineering", "technology": "Polars (streaming aggregation)",
     "evidence_file": PILLAR_DIRS["data_engineering"] / "train_features.csv"},
    {"layer": "Feature Engineering", "technology": "Polars (trend/delta/ratio/interaction features)",
     "evidence_file": PILLAR_DIRS["feature_engineering"] / "train_split_engineered.csv"},
    {"layer": "Model Training", "technology": "scikit-learn / XGBoost / LightGBM / CatBoost",
     "evidence_file": PILLAR_DIRS["model_development"] / "model_comparison.csv"},
    {"layer": "Explainability", "technology": "SHAP / LIME",
     "evidence_file": PILLAR_DIRS["explainable_ai"] / "notebook_06_summary.json"},
    {"layer": "Model Risk Governance", "technology": "PSI / rank-ordering / sensitivity validation",
     "evidence_file": PILLAR_DIRS["model_risk_management"] / "population_stability_index.csv"},
    {"layer": "Regulatory Mapping", "technology": "Basel III IRB (QRRE) / IFRS 9 staging",
     "evidence_file": PILLAR_DIRS["basel_ifrs9"] / "capital_adequacy_summary.csv"},
    {"layer": "MLOps", "technology": "Versioned model registry, GitHub Actions CI/CD",
     "evidence_file": MLOPS_DIR / "model_registry.json"},
    {"layer": "Model Serving", "technology": "FastAPI (self-tested via TestClient)",
     "evidence_file": PILLAR_DIRS["fastapi_deployment"] / "main.py"},
    {"layer": "Containerization", "technology": "Docker (python:3.11-slim, non-root)",
     "evidence_file": PILLAR_DIRS["docker"] / "Dockerfile"},
    {"layer": "Production Monitoring", "technology": "PSI + default-rate + AUC alerting, schedulable job",
     "evidence_file": PILLAR_DIRS["monitoring"] / "monitoring_job.py"},
    {"layer": "Business Intelligence", "technology": "Power BI star schema (CSV + Parquet) + DAX",
     "evidence_file": PILLAR_DIRS["powerbi_dashboard"] / "fact_customer_risk_scores.parquet"},
    {"layer": "Executive Reporting", "technology": "Financial impact model + interactive HTML dashboard",
     "evidence_file": PILLAR_DIRS["executive_reports"] / "Financial_Impact_Dashboard.html"},
]
_stack_rows = []
for _c in _evidence_checks:
    _present = _c["evidence_file"].exists()
    _stack_rows.append({"layer": _c["layer"], "technology": _c["technology"],
                         "evidence_file": _c["evidence_file"].name, "built": _present})

stack_df = pd.DataFrame(_stack_rows)
stack_path = PROD_ARCH_DIR / "technology_stack_inventory.csv"
stack_df.to_csv(stack_path, index=False)
print(stack_df.to_string(index=False))
print(f"\n{int(stack_df['built'].sum())} of {len(stack_df)} architecture layers have real, on-disk evidence this run.")
print(f"\u2705 Saved -> {stack_path}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: CAPACITY SIZING -- REAL MEASUREMENT + STATED SCALE ASSUMPTIONS
# =============================================================================
_section("SECTION 4: Capacity Sizing -- Real Measurement + Stated Scale Assumptions")

# --- ASSUMPTION: production-scale inputs the Kaggle dataset cannot supply.
#     Illustrative and editable -- broadly consistent with Notebook 14's own
#     stated deployment-scenario portfolio size, though this notebook does
#     not have a machine-readable link to that value (Notebook 14 does not
#     export it to its summary artifact), so this is restated here, not
#     silently assumed to match. ---
CAPACITY_SCALE_ASSUMPTIONS = {
    "deployment_portfolio_size_accounts": 2_000_000,        # ASSUMPTION -- illustrative mid-size issuer
    "daily_scoring_events_per_account": 0.05,                # ASSUMPTION -- ~1 score per account every 20 days (statement cycle proxy)
    "peak_to_average_load_ratio": 3.0,                        # ASSUMPTION -- month-end / campaign peak multiplier
    "sla_p99_latency_ms_target": 250.0,                       # ASSUMPTION -- a stated internal SLA target, not a regulatory figure
    "replica_safety_margin": 1.5,                             # ASSUMPTION -- headroom above bare peak-throughput requirement
}

_daily_events = (CAPACITY_SCALE_ASSUMPTIONS["deployment_portfolio_size_accounts"]
                  * CAPACITY_SCALE_ASSUMPTIONS["daily_scoring_events_per_account"])
_avg_qps = _daily_events / 86400.0
_peak_qps = _avg_qps * CAPACITY_SCALE_ASSUMPTIONS["peak_to_average_load_ratio"]

if LATEST_REGISTRY_ENTRY and LATEST_REGISTRY_ENTRY.get("batch_throughput_rows_per_sec"):
    _measured_throughput_per_replica = float(LATEST_REGISTRY_ENTRY["batch_throughput_rows_per_sec"])
    _measured_p99_ms = LATEST_REGISTRY_ENTRY.get("latency_p99_ms")
    _throughput_source = f"MEASURED -- Notebook 09's real batch benchmark, {REGISTRY_PATH.name}"
else:
    _measured_throughput_per_replica = None
    _measured_p99_ms = None
    _throughput_source = "NOT AVAILABLE -- Notebook 09 has not been run yet; no real throughput measurement exists"

if _measured_throughput_per_replica:
    _required_replicas = max(1, math.ceil(
        (_peak_qps * CAPACITY_SCALE_ASSUMPTIONS["replica_safety_margin"]) / _measured_throughput_per_replica))
else:
    _required_replicas = None

_sla_check = None
if _measured_p99_ms is not None:
    _sla_check = _measured_p99_ms <= CAPACITY_SCALE_ASSUMPTIONS["sla_p99_latency_ms_target"]

capacity_sizing = {
    "scale_assumptions": CAPACITY_SCALE_ASSUMPTIONS,
    "derived_avg_qps": round(_avg_qps, 4),
    "derived_peak_qps": round(_peak_qps, 4),
    "throughput_source": _throughput_source,
    "measured_throughput_rows_per_sec_per_replica": _measured_throughput_per_replica,
    "measured_p99_latency_ms": _measured_p99_ms,
    "sla_p99_target_ms": CAPACITY_SCALE_ASSUMPTIONS["sla_p99_latency_ms_target"],
    "sla_met_by_measured_latency": _sla_check,
    "required_replicas": _required_replicas,
}
capacity_sizing_path = PROD_ARCH_DIR / "capacity_sizing.json"
with open(capacity_sizing_path, "w", encoding="utf-8") as f:
    json.dump(capacity_sizing, f, indent=2)

print(f"Assumed portfolio scale       : {CAPACITY_SCALE_ASSUMPTIONS['deployment_portfolio_size_accounts']:,} accounts (ASSUMPTION)")
print(f"Derived average load          : {_avg_qps:.4f} requests/sec")
print(f"Derived peak load ({CAPACITY_SCALE_ASSUMPTIONS['peak_to_average_load_ratio']}x)  : {_peak_qps:.4f} requests/sec")
print(f"Per-replica throughput        : {_throughput_source}")
if _measured_throughput_per_replica:
    print(f"  Measured value               : {_measured_throughput_per_replica:,.0f} rows/sec")
    print(f"  Measured p99 latency          : {_measured_p99_ms:.3f} ms  (SLA target {CAPACITY_SCALE_ASSUMPTIONS['sla_p99_latency_ms_target']} ms -- "
          f"{'MET' if _sla_check else 'NOT MET'})")
    print(f"  Required replicas (with {CAPACITY_SCALE_ASSUMPTIONS['replica_safety_margin']}x safety margin): {_required_replicas}")
print(f"\u2705 Saved -> {capacity_sizing_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: TARGET PRODUCTION ARCHITECTURE -- TOPOLOGY & DIAGRAM
# =============================================================================
_section("SECTION 5: Target Production Architecture -- Topology & Diagram")

# --- Each component is tagged BUILT (real, this platform produced it) or
#     RECOMMENDED (a stated, sensible production addition this platform does
#     NOT implement -- e.g. Kubernetes orchestration, a managed feature
#     store). Never blurred. ---
ARCHITECTURE_COMPONENTS = [
    {"layer": "Ingestion", "component": "Batch/streaming statement ingestion", "status": "RECOMMENDED",
     "note": "This platform reads static Kaggle CSVs (Notebook 02); production needs a real ingestion pipeline"},
    {"layer": "Feature Store", "component": "Managed feature store", "status": "RECOMMENDED",
     "note": "This platform recomputes features per run (Notebook 04); production benefits from a served feature store"},
    {"layer": "Model Serving", "component": "FastAPI scoring service", "status": "BUILT",
     "note": "Notebook 10 -- real, self-tested against the champion model"},
    {"layer": "Model Serving", "component": "Container image", "status": "BUILT",
     "note": "Notebook 11 -- real Dockerfile, non-root, healthcheck"},
    {"layer": "Model Serving", "component": "Orchestration (Kubernetes/ECS)", "status": "RECOMMENDED",
     "note": "This platform delivers a single-container image; orchestration is an infra-team decision (Notebook 11)"},
    {"layer": "Model Serving", "component": "Load balancer / API gateway", "status": "RECOMMENDED",
     "note": "Needed to distribute traffic across replicas and terminate TLS"},
    {"layer": "MLOps", "component": "Model registry + CI/CD", "status": "BUILT",
     "note": "Notebook 09 -- real versioned registry, real GitHub Actions YAML"},
    {"layer": "Monitoring", "component": "Drift + performance alerting job", "status": "BUILT",
     "note": "Notebook 12 -- real, schedulable monitoring_job.py"},
    {"layer": "Monitoring", "component": "APM / centralized logging", "status": "RECOMMENDED",
     "note": "This platform's monitoring is model-quality focused, not infrastructure observability"},
    {"layer": "BI / Reporting", "component": "Power BI data model", "status": "BUILT",
     "note": "Notebook 13 -- real star schema export, real DAX"},
    {"layer": "BI / Reporting", "component": "Executive financial dashboard", "status": "BUILT",
     "note": "Notebook 14 -- real interactive HTML dashboard"},
    {"layer": "Security", "component": "Secrets manager / vault", "status": "RECOMMENDED",
     "note": "This platform's Dockerfile is checked for hardcoded secrets (Notebook 11) but does not integrate a vault"},
]
components_df = pd.DataFrame(ARCHITECTURE_COMPONENTS)
components_path = PROD_ARCH_DIR / "target_architecture_components.csv"
components_df.to_csv(components_path, index=False)
print(components_df.to_string(index=False))

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_green": "#3a9e5f", "cat_grey": "#9c9b96"}

import textwrap

_layer_order = []
for r in ARCHITECTURE_COMPONENTS:
    if r["layer"] not in _layer_order:
        _layer_order.append(r["layer"])
_layer_comps = {_layer: [r for r in ARCHITECTURE_COMPONENTS if r["layer"] == _layer] for _layer in _layer_order}

# --- Sized dynamically off the ACTUAL widest row, not a fixed guess -- a
#     fixed-width canvas silently clipped a box off-frame the first time
#     this was tested with a 4-component row, losing real information. ---
_box_w, _box_h, _box_gap, _label_col_w = 2.6, 0.7, 0.3, 2.8
_max_per_row = max(len(v) for v in _layer_comps.values())
_xlim_right = _label_col_w + _max_per_row * (_box_w + _box_gap) + 0.2
_fig_w = max(9.0, _xlim_right * 0.85)
_fig_h = max(6.0, len(_layer_order) * 0.95 + 1.5)

fig, ax = plt.subplots(figsize=(_fig_w, _fig_h), dpi=150)
ax.set_facecolor(VIZ["surface"]); fig.set_facecolor(VIZ["surface"])
ax.axis("off")
_y = len(_layer_order)
for _layer in _layer_order:
    _comps = _layer_comps[_layer]
    ax.text(0.2, _y - 0.5, _layer, ha="left", va="center", fontsize=9, color=VIZ["text_secondary"], weight="bold")
    for _i, _c in enumerate(_comps):
        _x = _label_col_w + _i * (_box_w + _box_gap)
        _color = VIZ["cat_green"] if _c["status"] == "BUILT" else VIZ["cat_grey"]
        ax.add_patch(plt.Rectangle((_x, _y - 0.85), _box_w, _box_h, facecolor=_color, edgecolor="none"))
        _wrapped = "\n".join(textwrap.wrap(_c["component"], width=20))
        ax.text(_x + _box_w / 2, _y - 0.5, _wrapped, ha="center", va="center", fontsize=7.5, color="white", weight="bold")
    _y -= 1.0
ax.set_xlim(0, _xlim_right); ax.set_ylim(0, len(_layer_order) + 0.3)
_legend_patches = [plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_green"], label="BUILT -- real, on disk this run"),
                    plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_grey"], label="RECOMMENDED -- not implemented by this platform")]
ax.legend(handles=_legend_patches, loc="lower right", fontsize=8, frameon=False)
ax.set_title(f"{PROBLEM_NAME}\nTarget Production Architecture -- Built vs. Recommended", fontsize=11, color=VIZ["text_primary"])
fig.tight_layout()
architecture_chart_path = PROD_ARCH_DIR / "production_architecture_diagram.png"
fig.savefig(architecture_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {components_path}")
print(f"\u2705 Saved -> {architecture_chart_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: HIGH AVAILABILITY, DISASTER RECOVERY & SECURITY (STATED POLICY)
# =============================================================================
_section("SECTION 6: High Availability, Disaster Recovery & Security (Stated Policy)")

# --- Operational policy this dataset cannot supply -- every line labeled
#     ASSUMPTION/POLICY, editable by the team that owns production. ---
HA_DR_POLICY = {
    "availability_target": "ASSUMPTION -- 99.9% monthly uptime for the scoring API (an internal SLA, not a regulatory figure).",
    "replica_policy": f"ASSUMPTION -- run at least 2 replicas at all times regardless of the Section 4 sizing calculation, for failover.",
    "multi_az_deployment": "ASSUMPTION -- deploy replicas across at least 2 availability zones.",
    "model_registry_backup": "ASSUMPTION -- Notebook 09's model_registry.json and every registered .joblib file should be backed up to durable, versioned storage (e.g. object storage with versioning enabled).",
    "rollback_procedure": "ASSUMPTION -- keep the prior container image tag available; a failed health check after deploy triggers an automatic rollback to the last known-good tag.",
    "rto_target": "ASSUMPTION -- Recovery Time Objective of 15 minutes for the scoring API.",
    "rpo_target": "ASSUMPTION -- Recovery Point Objective of 24 hours for the model registry (matches a daily retraining/registration cadence, if adopted).",
}
# --- The first two lines below are checked against Notebook 11's real,
#     on-disk dockerfile_lint_report.json -- never asserted without evidence. ---
_dockerfile_lint_path = PILLAR_DIRS["docker"] / "dockerfile_lint_report.json"
_dockerfile_lint = None
if _dockerfile_lint_path.exists():
    with open(_dockerfile_lint_path, "r", encoding="utf-8") as f:
        _dockerfile_lint = json.load(f)

if _dockerfile_lint:
    _non_root_line = (f"MEASURED -- Notebook 11's real Dockerfile lint: "
                       f"{'confirmed true' if _dockerfile_lint['checks'].get('runs_as_non_root') else 'FALSE -- review the Dockerfile'}.")
    _no_secrets_line = (f"MEASURED -- Notebook 11's real Dockerfile lint: "
                         f"{'confirmed true' if _dockerfile_lint['checks'].get('no_hardcoded_secrets') else 'FALSE -- review the Dockerfile'}.")
else:
    _non_root_line = "NOT VERIFIED -- Notebook 11 has not been run yet; no real lint evidence exists."
    _no_secrets_line = "NOT VERIFIED -- Notebook 11 has not been run yet; no real lint evidence exists."

SECURITY_POLICY = {
    "container_runs_as_non_root": _non_root_line,
    "no_hardcoded_secrets_in_image": _no_secrets_line,
    "secrets_management": "ASSUMPTION -- production should inject secrets (DB credentials, API keys) via a secrets manager/vault, never baked into the image or .env checked into source control.",
    "network_segmentation": "ASSUMPTION -- the scoring API should sit in a private subnet behind a load balancer, not directly internet-facing.",
    "transport_encryption": "ASSUMPTION -- TLS termination at the load balancer, minimum TLS 1.2.",
    "authentication": "ASSUMPTION -- this platform's main.py does not implement authentication (see Notebook 10) -- production must add an API key or OAuth2 layer before external exposure.",
    "audit_logging": "ASSUMPTION -- every /predict call should be logged with request ID, model version (Notebook 09), and predicted PD for audit trail purposes.",
}
print("High Availability / Disaster Recovery:")
for k, v in HA_DR_POLICY.items():
    print(f"  {k}: {v}")
print("\nSecurity:")
for k, v in SECURITY_POLICY.items():
    print(f"  {k}: {v}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: INFRASTRUCTURE COST ESTIMATE (STATED UNIT-COST ASSUMPTIONS)
# =============================================================================
_section("SECTION 7: Infrastructure Cost Estimate (Stated Unit-Cost Assumptions)")

INFRA_UNIT_COSTS = {
    "cost_per_replica_vcpu_hour_usd": 0.04,     # ASSUMPTION -- typical cloud on-demand small-instance vCPU rate
    "vcpus_per_replica": 1.0,                    # ASSUMPTION -- matches Notebook 11's single-process container
    "load_balancer_monthly_usd": 20.0,           # ASSUMPTION
    "monitoring_observability_monthly_usd": 150.0,  # ASSUMPTION
}
_replica_count_for_cost = _required_replicas if _required_replicas else 2  # fall back to the HA minimum if Notebook 09 hasn't run
_monthly_compute_cost = (INFRA_UNIT_COSTS["cost_per_replica_vcpu_hour_usd"]
                          * INFRA_UNIT_COSTS["vcpus_per_replica"] * 24 * 30 * _replica_count_for_cost)
_monthly_infra_total = (_monthly_compute_cost + INFRA_UNIT_COSTS["load_balancer_monthly_usd"]
                         + INFRA_UNIT_COSTS["monitoring_observability_monthly_usd"])

infra_cost_estimate = {
    "unit_cost_assumptions": INFRA_UNIT_COSTS,
    "replica_count_used": _replica_count_for_cost,
    "replica_count_source": "Section 4 real sizing calculation" if _required_replicas else "HA minimum fallback (Notebook 09 has not run -- no real sizing available)",
    "monthly_compute_cost_usd": round(_monthly_compute_cost, 2),
    "monthly_infra_total_usd": round(_monthly_infra_total, 2),
}
if NB14_SUMMARY and NB14_SUMMARY.get("monthly_recurring_cost_usd"):
    infra_cost_estimate["notebook_14_stated_monthly_recurring_cost_usd"] = NB14_SUMMARY["monthly_recurring_cost_usd"]
    infra_cost_estimate["comparison_note"] = (
        "Notebook 14's monthly recurring cost figure covers the FULL platform (staffing, tooling, licensing), "
        "not just infrastructure compute -- these two figures are not directly comparable, shown side-by-side for context only."
    )

infra_cost_path = PROD_ARCH_DIR / "infrastructure_cost_estimate.json"
with open(infra_cost_path, "w", encoding="utf-8") as f:
    json.dump(infra_cost_estimate, f, indent=2)
print(f"Estimated monthly infra cost (compute + LB + monitoring): ${_monthly_infra_total:,.2f}  "
      f"(using {_replica_count_for_cost} replicas -- {infra_cost_estimate['replica_count_source']})")
print(f"\u2705 Saved -> {infra_cost_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: PRODUCTION READINESS CHECKLIST
# =============================================================================
_section("SECTION 8: Production Readiness Checklist")

production_checklist = [
    {"dimension": "Technology Stack As-Built Documented", "status": "Pass",
     "evidence": f"{int(stack_df['built'].sum())}/{len(stack_df)} layers verified on disk"},
    {"dimension": "Capacity Sizing Computed", "status": "Pass" if _measured_throughput_per_replica else "Fallback (Notebook 09 not yet run)",
     "evidence": capacity_sizing["throughput_source"]},
    {"dimension": "SLA Latency Target Checked Against Real Measurement", "status": (
        "Pass" if _sla_check else ("Review Needed" if _sla_check is False else "Not Verified in This Environment")),
     "evidence": f"measured p99 {_measured_p99_ms} ms vs target {CAPACITY_SCALE_ASSUMPTIONS['sla_p99_latency_ms_target']} ms" if _measured_p99_ms else "no measurement available"},
    {"dimension": "Target Architecture Diagram (Built vs. Recommended)", "status": "Pass",
     "evidence": f"{len(components_df)} components, {int((components_df['status']=='BUILT').sum())} built"},
    {"dimension": "HA / DR Policy Documented", "status": "Pass", "evidence": f"{len(HA_DR_POLICY)} policy items (ASSUMPTION)"},
    {"dimension": "Security Policy Documented", "status": "Pass", "evidence": f"{len(SECURITY_POLICY)} policy items"},
    {"dimension": "Infrastructure Cost Estimated", "status": "Pass", "evidence": f"${_monthly_infra_total:,.2f}/month (ASSUMPTION unit costs)"},
    {"dimension": "Orchestration (Kubernetes/ECS) Implemented", "status": "Not Yet Completed",
     "evidence": "RECOMMENDED, out of this platform's scope -- see Notebook 11"},
    {"dimension": "Secrets Manager Integrated", "status": "Not Yet Completed", "evidence": "RECOMMENDED, out of this platform's scope"},
]
production_checklist_df = pd.DataFrame(production_checklist)
production_checklist_path = PROD_ARCH_DIR / "production_readiness_checklist.csv"
production_checklist_df.to_csv(production_checklist_path, index=False)
print(production_checklist_df.to_string(index=False))
print(f"\u2705 Saved -> {production_checklist_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WORD REPORT -- PRODUCTION_ARCHITECTURE_REPORT.DOCX
# =============================================================================
_section("SECTION 9: Word Report -- Production_Architecture_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = str(v)
    return table


def _add_table_from_df(doc, df, max_rows=25):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Production Architecture Report -- Notebook 16")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(report, "1. Technology Stack -- As-Built", level=1)
report.add_paragraph("Every row verified against a real file this platform actually wrote to disk this run.")
_add_table_from_df(report, stack_df)

_add_heading(report, "2. Capacity Sizing", level=1)
report.add_paragraph(
    "Scale inputs (portfolio size, daily scoring rate, peak ratio, SLA target) are stated, editable "
    "ASSUMPTIONs -- the dataset has no ground truth for production request volume. Throughput and latency, "
    "where shown, are Notebook 09's real, measured benchmark, never estimated."
)
_add_kv_table(report, CAPACITY_SCALE_ASSUMPTIONS)
_add_kv_table(report, {
    "derived_avg_qps": round(_avg_qps, 4), "derived_peak_qps": round(_peak_qps, 4),
    "throughput_source": capacity_sizing["throughput_source"],
    "required_replicas": _required_replicas if _required_replicas else "N/A -- Notebook 09 has not run",
})

_add_heading(report, "3. Target Production Architecture", level=1)
report.add_picture(str(architecture_chart_path), width=Inches(6.3))
_add_table_from_df(report, components_df)

_add_heading(report, "4. High Availability & Disaster Recovery (Stated Policy)", level=1)
_add_kv_table(report, HA_DR_POLICY)

_add_heading(report, "5. Security (Stated Policy)", level=1)
_add_kv_table(report, SECURITY_POLICY)

_add_heading(report, "6. Infrastructure Cost Estimate", level=1)
_add_kv_table(report, {k: v for k, v in infra_cost_estimate.items() if k != "unit_cost_assumptions"})
_add_kv_table(report, INFRA_UNIT_COSTS)

_add_heading(report, "7. Production Readiness Checklist", level=1)
_add_table_from_df(report, production_checklist_df)

report_path = PROD_ARCH_DIR / "Production_Architecture_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 10: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Technology stack inventory covers all 12 layers", len(stack_df) == 12, f"({len(stack_df)})")
_check("Capacity sizing JSON honestly reflects Notebook 09's run status",
       (capacity_sizing["measured_throughput_rows_per_sec_per_replica"] is not None) == (LATEST_REGISTRY_ENTRY is not None))
_check("Target architecture has both BUILT and RECOMMENDED components",
       set(components_df["status"]) == {"BUILT", "RECOMMENDED"})
_check("Infra cost estimate is positive", _monthly_infra_total > 0, f"({_monthly_infra_total})")
_check("Production checklist covers 9 dimensions", len(production_checklist_df) == 9, f"({len(production_checklist_df)})")

_expected_files = [stack_path, capacity_sizing_path, components_path, architecture_chart_path,
                    infra_cost_path, production_checklist_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 16 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 16 checks passed.")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 11: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
}
performance_report_path = ARTIFACTS_DIR / "notebook_16_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 16 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 12: Write Notebook 16 Summary Artifact")

notebook_16_summary = {
    "notebook": "16_production_architecture", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "layers_built": int(stack_df["built"].sum()), "layers_total": int(len(stack_df)),
    "required_replicas": _required_replicas, "sla_met_by_measured_latency": _sla_check,
    "monthly_infra_cost_usd": round(_monthly_infra_total, 2),
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb16_summary_path = ARTIFACTS_DIR / "notebook_16_summary.json"
with open(nb16_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_16_summary, f, indent=2)
print(f"\u2705 Saved -> {nb16_summary_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 13: Notebook 16 Complete -- Handoff to Notebook 17")

print("NOTEBOOK 16: PRODUCTION ARCHITECTURE -- COMPLETE")
print(f"  Technology stack layers built    : {int(stack_df['built'].sum())} / {len(stack_df)}")
print(f"  Required replicas (real sizing)   : {_required_replicas if _required_replicas else 'N/A -- Notebook 09 has not run'}")
print(f"  SLA met by measured latency       : {_sla_check if _sla_check is not None else 'N/A'}")
print(f"  Estimated monthly infra cost       : ${_monthly_infra_total:,.2f}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb16_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 17_comprehensive_reporting.ipynb")
print("\n\u2705 Ready to proceed.")
